# Noisy GNLSE Propagation — Reparameterization Trick & FDT Balance

Tests `gnlse_solver_noisy.py`. Covers:

1. **Imports & physical parameters**
2. **Noiseless N=1 baseline** — saved to `figures/`
3. **Sigma sweep** — single trajectory at several noise levels
4. **Ensemble statistics** — mean ± std over realisations
5. **FDT balance demo** — pure ASE (energy grows) vs balanced loss (stationary E)
6. **Gradient test** — reparameterization via finite differences
7. **FD projection check**
8. **Loss-vs-sigma curve** — smooth, differentiable
9. **Coloured noise demo** — spectral shaping + step-size invariance

All figures saved under `figures/`.


In [ ]:
import os
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import time as time_mod

from gnlse_solver_noisy import (
    make_args,
    make_noise_samples,
    GNLSE3D_propagate,
    GNLSE3D_propagate_noisy,
    windowed_forward_noisy,
    windowed_grad_noisy,
    make_windowed_context_noisy,
    make_noise_filter,
    _prepare_propagation,
    _resolve_precision,
)

jax.config.update('jax_enable_x64', True)
print('JAX devices:', jax.devices())

os.makedirs('figures', exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 9})
print('figures/ directory ready.')


In [ ]:
# ── Physical parameters (same as transfer_operator_1d_ist.ipynb) ──────────────
c0      = 2.998e8
lambda0 = 1064e-9
n_ref   = 1.453
n2      = 2.76e-20
# beta2 > 0 = anomalous in solver convention (see CLAUDE.md)
beta2   = +22e-27

omega0  = 2 * np.pi * c0 / lambda0
gamma   = n2 * omega0 / c0

T0      = 100e-15               # 100 fs sech width
L_D     = T0**2 / beta2
I_sol   = beta2 / (gamma * T0**2)   # N=1 soliton peak intensity [W/m^2]

print(f'gamma  = {gamma:.4e} m/W')
print(f'L_D    = {L_D*1e3:.4f} mm')
print(f'I_sol  = {I_sol:.3e} W/m^2  = peak |A|^2 for N=1')
print(f'sqrt(I_sol) = {np.sqrt(I_sol):.3e}  = peak |A| [sqrt(W/m^2)]')


In [ ]:
# ── Grid ──────────────────────────────────────────────────────────────────────
Nt   = 1024
Lt   = 10e-12        # 10 ps window
dt   = Lt / Nt
t    = np.arange(Nt) * dt
t_c  = Lt / 2
t_ps = t * 1e12

z_period = np.pi / 2 * L_D
Lz      = 3 * z_period
deltaZ  = L_D / 100
n_steps = int(round(Lz / deltaZ))
n_saves = 60

args = make_args(
    Nx=1, Ny=1, Nt=Nt,
    Lx=50e-6, Ly=50e-6,
    Lt=Lt, Lz=Lz,
    n_ref=n_ref, n2_val=n2,
    lambda0=lambda0, beta2_val=beta2,
    deltaZ=deltaZ,
    fr=0.0, sw=0, pml_thickness=0,
    precision='fp64', n_saves=n_saves,
)
print(f'Nt={Nt}, Lt={Lt*1e12:.0f} ps, T0={T0*1e15:.0f} fs')
print(f'Lz = {Lz*1e3:.3f} mm = {Lz/z_period:.0f} soliton periods')
print(f'Steps: {n_steps}, saves: {n_saves}')

# N=1 soliton IC
A_N1    = (np.sqrt(I_sol) / np.cosh((t - t_c) / T0)).astype(np.complex128)
A_N1_3d = A_N1[np.newaxis, np.newaxis, :]
E_sol_num = float(np.sum(np.abs(A_N1)**2) * dt)
z_mm    = np.linspace(0, Lz*1e3, n_saves)

# Noiseless baseline
print('\nPropagating noiseless N=1 soliton...')
t0 = time_mod.time()
res_clean = GNLSE3D_propagate_noisy(args, A_N1_3d, eps=None, noise_sigma=0.0, save_as_fp32=False)
print(f'Done in {time_mod.time()-t0:.1f}s')
field_clean = np.array(res_clean['field'][0, 0, :, :])   # (Nt, n_saves)
peak_clean  = np.max(np.abs(field_clean)**2, axis=0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].pcolormesh(z_mm, t_ps, np.abs(field_clean)**2/I_sol,
                   shading='auto', cmap='inferno')
axes[0].set_ylim(t_ps[Nt//4], t_ps[3*Nt//4])
axes[0].set_xlabel('z (mm)'); axes[0].set_ylabel('t (ps)')
axes[0].set_title('Noiseless N=1 soliton: z-t map')

axes[1].plot(z_mm, peak_clean/I_sol)
axes[1].axhline(1.0, color='gray', ls='--')
axes[1].set_xlabel('z (mm)'); axes[1].set_ylabel('Peak |A|²/I_sol')
axes[1].set_title(f'Peak stability (σ = {np.std(peak_clean/I_sol)*100:.3f}%)')
axes[1].grid(alpha=0.3)

axes[2].plot(t_ps, np.abs(field_clean[:,0])**2/I_sol,  label='input')
axes[2].plot(t_ps, np.abs(field_clean[:,-1])**2/I_sol, '--', label='output')
axes[2].set_xlim(t_ps[Nt//4], t_ps[3*Nt//4])
axes[2].set_xlabel('t (ps)'); axes[2].legend()
axes[2].set_title('Input vs output'); axes[2].grid(alpha=0.3)

plt.suptitle('Noiseless baseline', fontsize=11)
plt.tight_layout()
plt.savefig('figures/01_noiseless_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/01_noiseless_baseline.png')


In [ ]:
# ── Single noisy trajectory at several σ values (pure ASE, loss_coeff=0) ──────
prep        = _prepare_propagation(args, A_N1_3d)
steps_total = prep['steps_total']
print(f'steps_total = {steps_total}')

sigma_fracs = [0.0, 0.01, 0.05, 0.20]
key0        = jax.random.PRNGKey(42)
eps         = make_noise_samples(key0, steps_total, 1, 1, Nt, dtype=jnp.complex128)
print(f'eps: mean |eps|² = {float(jnp.mean(jnp.abs(eps)**2)):.4f}  (expect 1.0)')

fig, axes = plt.subplots(len(sigma_fracs), 2, figsize=(13, 3.5*len(sigma_fracs)),
                         gridspec_kw=dict(wspace=0.35, hspace=0.5))

for row, frac in enumerate(sigma_fracs):
    sigma = frac * np.sqrt(I_sol)
    res   = GNLSE3D_propagate_noisy(args, A_N1_3d, eps, sigma,
                                    loss_coeff=0.0, save_as_fp32=False)
    f     = np.array(res['field'][0, 0, :, :])

    im = axes[row,0].pcolormesh(z_mm, t_ps, np.abs(f)**2/I_sol,
                                 shading='auto', cmap='inferno', vmin=0)
    axes[row,0].set_ylim(t_ps[Nt//4], t_ps[3*Nt//4])
    axes[row,0].set_xlabel('z (mm)'); axes[row,0].set_ylabel('t (ps)')
    axes[row,0].set_title(f'σ = {frac:.0%}√I_sol  (ASE only)', fontsize=9)
    plt.colorbar(im, ax=axes[row,0], label='|A|²/I_sol')

    axes[row,1].plot(t_ps, np.abs(f[:,0])**2/I_sol,  'C0', lw=1.5, label='input')
    axes[row,1].plot(t_ps, np.abs(f[:,-1])**2/I_sol, 'C1--', lw=1.5, label='output')
    axes[row,1].set_xlim(t_ps[Nt//4], t_ps[3*Nt//4])
    peak_out = np.max(np.abs(f[:,-1])**2)
    axes[row,1].set_title(f'Peak_out/I_sol = {peak_out/I_sol:.3f}', fontsize=9)
    axes[row,1].set_xlabel('t (ps)'); axes[row,1].legend(fontsize=8)
    axes[row,1].grid(alpha=0.3)

plt.suptitle('Pure ASE noise (loss_coeff=0): effect of σ on N=1 soliton (same ε realisation)', fontsize=10)
plt.tight_layout()
plt.savefig('figures/02_sigma_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/02_sigma_sweep.png')


In [ ]:
# ── Ensemble: mean ± std at fixed σ ──────────────────────────────────────────
sigma_frac = 0.05
sigma      = sigma_frac * np.sqrt(I_sol)
N_ens      = 20
key_ens    = jax.random.PRNGKey(1337)

I_out_ensemble = np.zeros((N_ens, Nt))
peak_ens       = np.zeros((N_ens, n_saves))

print(f'Running {N_ens} trajectories at σ={sigma_frac:.0%}√I_sol (ASE only)...')
t0 = time_mod.time()
for n in range(N_ens):
    key_ens, subkey = jax.random.split(key_ens)
    eps_n = make_noise_samples(subkey, steps_total, 1, 1, Nt, dtype=jnp.complex128)
    res_n = GNLSE3D_propagate_noisy(args, A_N1_3d, eps_n, sigma,
                                    loss_coeff=0.0, save_as_fp32=False)
    f_n   = np.array(res_n['field'][0, 0, :, :])
    I_out_ensemble[n] = np.abs(f_n[:, -1])**2
    peak_ens[n]       = np.max(np.abs(f_n)**2, axis=0)
print(f'{N_ens} trajectories in {time_mod.time()-t0:.1f}s')

I_mean = np.mean(I_out_ensemble, axis=0)
I_std  = np.std( I_out_ensemble, axis=0)
I_ref  = np.abs(field_clean[:, -1])**2

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ax = axes[0]
ax.fill_between(t_ps, (I_mean-I_std)/I_sol, (I_mean+I_std)/I_sol,
                alpha=0.35, color='C0', label='mean ± std')
ax.plot(t_ps, I_mean/I_sol, 'C0', lw=1.5)
ax.plot(t_ps, I_ref/I_sol,  'k--', lw=1.5, alpha=0.6, label='noiseless')
ax.set_xlim(t_ps[Nt//4], t_ps[3*Nt//4])
ax.set_xlabel('t (ps)'); ax.set_ylabel('|A|²/I_sol')
ax.set_title(f'Output: σ={sigma_frac:.0%}√I_sol, N={N_ens}')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
for n in range(N_ens):
    ax.plot(z_mm, peak_ens[n]/I_sol, 'C0', alpha=0.25, lw=0.8)
ax.plot(z_mm, np.mean(peak_ens,axis=0)/I_sol, 'C1', lw=2, label='ensemble mean')
ax.plot(z_mm, peak_clean/I_sol, 'k--', lw=1.5, label='noiseless')
ax.axhline(1.0, color='gray', ls=':', alpha=0.5)
ax.set_xlabel('z (mm)'); ax.set_ylabel('Peak |A|²/I_sol')
ax.set_title('Peak intensity vs z')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
snr = I_mean**2 / (I_std**2 + 1e-30)
ax.semilogy(t_ps, snr)
ax.set_xlim(t_ps[Nt//4], t_ps[3*Nt//4])
ax.set_xlabel('t (ps)'); ax.set_ylabel('SNR = μ²/σ²')
ax.set_title('Signal-to-noise at output'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/03_ensemble.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/03_ensemble.png')


In [ ]:
# ── Fluctuation-Dissipation Balance ───────────────────────────────────────────
#
# Pure additive ASE (loss_coeff=0):
#   dE/dz = sigma² · Nt · dt   →  E(z) = E_sol + sigma² · Nt · dt · z  (linear growth)
#
# FDT-balanced (loss_coeff = sigma²/2):
#   dA/dz = F(A) − loss_coeff·A + sigma·xi
#   Equilibrium:  E_eq = Nt·dt·1  (unit variance per mode)
#   For the soliton (nonlinear attractor) this pins E ≈ E_sol on average —
#   the noise temperature is stationary and independent of propagation length.
#
# The FDT-balanced model is the right one for coherent structure discovery:
# it gives a fixed "temperature" so σ_k values are interpretable as structural
# stability at a well-defined noise level, not a z-dependent one.

sigma_fracs_fdt = [0.03, 0.07, 0.15]
N_ens_fdt = 10
key_fdt   = jax.random.PRNGKey(271828)

fig, axes = plt.subplots(len(sigma_fracs_fdt), 2,
                         figsize=(14, 4.2*len(sigma_fracs_fdt)),
                         gridspec_kw=dict(wspace=0.38, hspace=0.55))

for row, frac in enumerate(sigma_fracs_fdt):
    sigma  = frac * np.sqrt(I_sol)
    lc_bal = float(sigma)**2 / 2.0    # FDT balance: loss_coeff = sigma²/2

    e_ase = np.zeros((N_ens_fdt, n_saves))
    e_fdt = np.zeros((N_ens_fdt, n_saves))
    prof_ase, prof_fdt = [], []

    for n in range(N_ens_fdt):
        key_fdt, subkey = jax.random.split(key_fdt)
        eps_n = make_noise_samples(subkey, steps_total, 1, 1, Nt, dtype=jnp.complex128)

        r_ase = GNLSE3D_propagate_noisy(args, A_N1_3d, eps_n, sigma,
                                        loss_coeff=0.0, save_as_fp32=False)
        f_ase = np.array(r_ase['field'][0, 0, :, :])
        e_ase[n] = np.sum(np.abs(f_ase)**2, axis=0) * dt
        prof_ase.append(np.abs(f_ase[:, -1])**2)

        r_fdt = GNLSE3D_propagate_noisy(args, A_N1_3d, eps_n, sigma,
                                        loss_coeff=lc_bal, save_as_fp32=False)
        f_fdt = np.array(r_fdt['field'][0, 0, :, :])
        e_fdt[n] = np.sum(np.abs(f_fdt)**2, axis=0) * dt
        prof_fdt.append(np.abs(f_fdt[:, -1])**2)

    # ── Left: energy vs z ─────────────────────────────────────────────────
    ax = axes[row, 0]
    for n in range(N_ens_fdt):
        ax.plot(z_mm, e_ase[n]/E_sol_num, color='C1', alpha=0.18, lw=0.7)
        ax.plot(z_mm, e_fdt[n]/E_sol_num, color='C0', alpha=0.18, lw=0.7)
    ax.plot(z_mm, e_ase.mean(0)/E_sol_num, 'C1', lw=2.2, label='ASE (loss=0)')
    ax.plot(z_mm, e_fdt.mean(0)/E_sol_num, 'C0', lw=2.2, label='FDT (loss=σ²/2)')
    # Analytical ASE drift
    z_arr    = np.array(res_clean['field'][0,0,:,:]).shape   # just need n_saves
    z_arr    = np.linspace(0, Lz, n_saves)
    e_theory = (E_sol_num + float(sigma)**2 * Nt * dt * z_arr) / E_sol_num
    ax.plot(z_mm, e_theory, 'C1--', lw=1.3, alpha=0.8, label='theory: E₀+σ²·Nₜ·dt·z')
    ax.axhline(1.0, color='k', ls=':', lw=1.0, alpha=0.4)
    ax.set_xlabel('z (mm)'); ax.set_ylabel('E(z) / E_sol')
    ax.set_title(f'σ = {frac:.0%}√I_sol  |  loss_coeff = {lc_bal:.2e} m⁻¹')
    ax.legend(fontsize=7.5); ax.grid(alpha=0.3)

    # ── Right: output intensity (mean ± std) ──────────────────────────────
    ax = axes[row, 1]
    I_ase = np.array(prof_ase) / I_sol
    I_fdt = np.array(prof_fdt) / I_sol
    I_ref = np.abs(field_clean[:, -1])**2 / I_sol
    ax.fill_between(t_ps, I_ase.mean(0)-I_ase.std(0), I_ase.mean(0)+I_ase.std(0),
                    color='C1', alpha=0.25)
    ax.fill_between(t_ps, I_fdt.mean(0)-I_fdt.std(0), I_fdt.mean(0)+I_fdt.std(0),
                    color='C0', alpha=0.25)
    ax.plot(t_ps, I_ase.mean(0), 'C1', lw=1.8, label='ASE mean')
    ax.plot(t_ps, I_fdt.mean(0), 'C0', lw=1.8, label='FDT mean')
    ax.plot(t_ps, I_ref,         'k--', lw=1.2, alpha=0.55, label='noiseless')
    ax.set_xlim(t_ps[Nt//4], t_ps[3*Nt//4])
    ax.set_xlabel('t (ps)'); ax.set_ylabel('|A|²/I_sol')
    ax.set_title(f'Output profile  ({N_ens_fdt} realisations, mean ± std)')
    ax.legend(fontsize=7.5); ax.grid(alpha=0.3)

plt.suptitle(
    'Fluctuation-Dissipation Balance\n'
    'ASE: energy grows linearly with z.  FDT (loss=σ²/2): E stationary at E_sol.',
    fontsize=11)
plt.tight_layout()
plt.savefig('figures/04_fdt_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/04_fdt_balance.png')
print()
print('FDT summary:')
for frac in sigma_fracs_fdt:
    sigma = frac * np.sqrt(I_sol)
    lc    = sigma**2 / 2
    L_damp = 1.0 / (2*lc) if lc > 0 else float('inf')
    print(f'  σ={frac:.0%}√I_sol: loss_coeff={lc:.3e} m⁻¹, '
          f'L_damp={L_damp*1e3:.1f} mm  ({L_damp/L_D:.1f} × L_D)')


In [ ]:
# ── Gradient test: verify reparameterization via finite differences ───────────
#
# Loss: L = sum |A_final(t)|^2  (total output energy)
# Test: analytical dL/dA0 from windowed_grad_noisy vs finite differences.
#
# We use a short propagation (1 soliton period) and few windows for speed.

# Short args for fast FD test
Lz_test = 0.5 * z_period     # half soliton period
args_test = make_args(
    Nx=1, Ny=1, Nt=Nt,
    Lx=50e-6, Ly=50e-6,
    Lt=Lt, Lz=Lz_test,
    n_ref=n_ref, n2_val=n2,
    lambda0=lambda0, beta2_val=beta2,
    deltaZ=deltaZ,
    fr=0.0, sw=0,
    pml_thickness=0,
    precision='fp64',
    n_saves=10,
)

prep_test   = _prepare_propagation(args_test, A_N1_3d)
steps_test  = prep_test['steps_total']
print(f'Test steps: {steps_test}')

key_test = jax.random.PRNGKey(999)
eps_test = make_noise_samples(key_test, steps_test, 1, 1, Nt, dtype=jnp.complex128)
sigma_test = 0.03 * np.sqrt(I_sol)

# Build context once
ctx_test = make_windowed_context_noisy(args_test, Nt, loss_coeff=0.0)
# loss_coeff=0 for gradient test: FDT damping is differentiable but we test the base case

# Loss in k-space: L = ||A_final_kwo||^2  (proportional to Parseval energy)
def loss_fn(field_kwo_final):
    return jnp.sum(jnp.abs(field_kwo_final)**2).real

n_win = 4
print(f'Computing analytical gradient (n_windows={n_win})...')
res_grad = windowed_grad_noisy(
    loss_fn, args_test, A_N1_3d, eps_test, sigma_test,
    n_windows=n_win, ctx=ctx_test,
)
grad_analytical = np.array(res_grad['grad'])   # (1, 1, Nt) complex kwo-space
print(f'Loss = {float(res_grad["loss"]):.6e}')
print(f'Grad norm = {np.linalg.norm(grad_analytical):.6e}')
print(f'fwd: {res_grad["fwd_seconds"]:.1f}s  bwd: {res_grad["bwd_seconds"]:.1f}s')


In [ ]:
# ── Finite-difference check on a random projection ────────────────────────────
# We test dL/dA0_kwo along a random direction v.
# JAX complex gradient convention: for real f, FD = Re(g · v)  (no conjugate, no factor 2).

key_fd = jax.random.PRNGKey(2025)
v_rand = np.array(jax.random.normal(key_fd, A_N1_3d.shape, dtype=jnp.float64), dtype=np.complex128)
v_rand /= np.linalg.norm(v_rand)

def compute_loss_at(A0_perturbed):
    fwd = windowed_forward_noisy(
        args_test, A0_perturbed, eps_test, sigma_test,
        n_windows=n_win, ctx=ctx_test, save_as_fp32=False
    )
    kwo = jnp.fft.fftn(fwd['field_final'], axes=(0,1,2))
    return float(loss_fn(kwo))

h_vals = [1e-1, 1e-2, 5e-3]
# v in k-space (FFT of v_xyt)
v_kwo = np.fft.fftn(v_rand, axes=(0,1,2))
# Analytical directional derivative: Re(g · v_kwo) — JAX convention for complex grads
directional_analytical = np.real(np.sum(grad_analytical * v_kwo))

print(f'Analytical directional derivative: {directional_analytical:.6e}')
print()

for h in h_vals:
    Lp = compute_loss_at(A_N1_3d + h * v_rand)
    Lm = compute_loss_at(A_N1_3d - h * v_rand)
    fd = (Lp - Lm) / (2 * h)
    rel_err = abs(fd - directional_analytical) / (abs(directional_analytical) + 1e-30)
    print(f'h={h:.0e}:  FD = {fd:.6e}   rel_err = {rel_err:.2e}')


In [ ]:
# ── Gradient w.r.t. noise_sigma via finite differences ───────────────────────
# Demonstrate that the reparameterization makes dL/d(sigma) well-defined.
# We perturb sigma directly and check the numerical derivative.

def loss_vs_sigma(sigma_val):
    """Scalar loss as a function of sigma (all else fixed)."""
    fwd = windowed_forward_noisy(
        args_test, A_N1_3d, eps_test, float(sigma_val),
        n_windows=n_win, ctx=ctx_test, save_as_fp32=False
    )
    kwo = jnp.fft.fftn(fwd['field_final'], axes=(0,1,2))
    return float(loss_fn(kwo))

sigma_vals  = np.linspace(0.0, 0.1 * np.sqrt(I_sol), 30)
loss_curve  = [loss_vs_sigma(s) for s in sigma_vals]
print('Loss vs sigma curve computed.')

# Finite-difference slope at sigma_test
delta_s = sigma_test * 0.01
dL_ds_fd = (loss_vs_sigma(sigma_test + delta_s) - loss_vs_sigma(sigma_test - delta_s)) / (2*delta_s)

# Analytical slope from the reparameterization:
# d/d(sigma) L(A0 + sigma*eps) = <grad_A L, eps_sum>
# where eps_sum = sum over steps of the propagated eps contribution.
# This is NOT trivially computable without a second backward pass;
# we just verify the curve is smooth and the FD slope is finite.
print(f'dL/d(sigma) at sigma_test (FD) = {dL_ds_fd:.4e}  [finite, gradient is usable]')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(sigma_vals / np.sqrt(I_sol) * 100, loss_curve, 'C0o-', ms=3)
axes[0].axvline(sigma_test/np.sqrt(I_sol)*100, color='C1', ls='--', label='σ_test')
axes[0].set_xlabel('σ / √I_sol  (%)')
axes[0].set_ylabel('Loss = ||A_final||²_k')
axes[0].set_title('Loss is smooth and differentiable w.r.t. σ (reparameterization)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Overlay several noise-level trajectories final profiles
key_disp = jax.random.PRNGKey(77)
eps_disp = make_noise_samples(key_disp, steps_test, 1, 1, Nt, dtype=jnp.complex128)
frac_vals = [0.0, 0.02, 0.05, 0.10]
for frac in frac_vals:
    fwd = windowed_forward_noisy(
        args_test, A_N1_3d, eps_disp, frac*np.sqrt(I_sol),
        n_windows=n_win, ctx=ctx_test, save_as_fp32=False
    )
    Af = np.array(fwd['field_final'][0, 0, :])
    axes[1].plot(t_ps, np.abs(Af)**2/I_sol, label=f'σ={frac:.0%}√I_sol')
axes[1].set_xlim(t_ps[Nt//4], t_ps[3*Nt//4])
axes[1].set_xlabel('t (ps)'); axes[1].set_ylabel('|A_final|²/I_sol')
axes[1].set_title('Output profiles at varying σ (windowed_forward_noisy)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/05_loss_vs_sigma.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Coloured noise demo ──────────────────────────────────────────────────────
# Demonstrates:
#  1. make_noise_filter shapes the noise power spectrum as expected
#  2. sqrt(deltaZ) fix: accumulated noise power is step-size invariant

from gnlse_solver_noisy import make_noise_filter

# ── 1. Build and visualise filter shapes ─────────────────────────────────────
freq_axis = np.fft.fftfreq(Nt, dt) * 1e-12   # THz, fftfreq ordering
freq_ps   = np.fft.fftshift(freq_axis)         # shift for plotting

filters = {
    'white (no filter)':         None,
    'Gaussian 1 THz':   make_noise_filter(Nt, dt, bandwidth_hz=1e12,  filter_type='gaussian'),
    'Gaussian 5 THz':   make_noise_filter(Nt, dt, bandwidth_hz=5e12,  filter_type='gaussian'),
    'Lorentzian 2 THz': make_noise_filter(Nt, dt, bandwidth_hz=2e12,  filter_type='lorentzian'),
    'Rect ±3 THz':      make_noise_filter(Nt, dt, bandwidth_hz=3e12,  filter_type='rect'),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for label, H in filters.items():
    if H is None:
        H_plot = np.ones(Nt)
    else:
        H_plot = np.array(H)
    axes[0].plot(freq_ps, np.fft.fftshift(H_plot), label=label)

axes[0].set_xlabel('Frequency (THz)'); axes[0].set_ylabel('H(f)  [normalised]')
axes[0].set_title('Colour filter shapes'); axes[0].legend(fontsize=8)
axes[0].axhline(1, color='gray', ls=':', alpha=0.4); axes[0].grid(alpha=0.3)

# ── 2. Measure output noise power spectrum for each filter ────────────────────
# Propagate noiseless once to get the clean field at every save point.
# Then subtract from noisy runs to isolate the noise contribution.
sigma_demo  = 0.04 * np.sqrt(I_sol)    # 4% of sqrt(I_sol)
key_demo    = jax.random.PRNGKey(5555)
eps_demo    = make_noise_samples(key_demo, steps_total, 1, 1, Nt, dtype=jnp.complex128)

for label, H in filters.items():
    res = GNLSE3D_propagate_noisy(
        args, A_N1_3d, eps_demo, sigma_demo,
        loss_coeff=0.0, noise_filter_w=H, save_as_fp32=False,
    )
    A_out = np.array(res['field'][0, 0, :, -1])
    A_ref = field_clean[:, -1]
    delta_A   = A_out - A_ref                        # noise contribution at output
    psd       = np.abs(np.fft.fftshift(np.fft.fft(delta_A)))**2
    axes[1].plot(freq_ps, 10*np.log10(psd / psd.max() + 1e-12), label=label)

axes[1].set_xlabel('Frequency (THz)'); axes[1].set_ylabel('Noise PSD (dB, normalised)')
axes[1].set_title('Output noise spectrum — matches filter shape'); axes[1].set_ylim(-60, 5)
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.suptitle('make_noise_filter: spectral shaping of per-step noise', fontsize=11)
plt.tight_layout()
plt.savefig('figures/06_coloured_noise_filters.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/06_coloured_noise_filters.png')

# ── 3. Step-size invariance check ────────────────────────────────────────────
# Same sigma, halved deltaZ: total noise power at output should be ~equal.
print('\n── Step-size invariance (sqrt(deltaZ) fix) ──')
sigma_inv = 0.03 * np.sqrt(I_sol)

for dZ_factor, label in [(1, 'nominal deltaZ'), (2, 'deltaZ / 2')]:
    dZ_here  = deltaZ / dZ_factor
    args_inv = make_args(
        Nx=1, Ny=1, Nt=Nt, Lx=50e-6, Ly=50e-6,
        Lt=Lt, Lz=Lz, n_ref=n_ref, n2_val=n2,
        lambda0=lambda0, beta2_val=beta2,
        deltaZ=dZ_here, fr=0.0, sw=0, pml_thickness=0,
        precision='fp64', n_saves=n_saves,
    )
    prep_inv  = _prepare_propagation(args_inv, A_N1_3d)
    N_inv     = prep_inv['steps_total']
    key_inv   = jax.random.PRNGKey(7777)
    eps_inv   = make_noise_samples(key_inv, N_inv, 1, 1, Nt, dtype=jnp.complex128)

    # Run 5 independent realisations and average noise energy at output
    energies = []
    for trial in range(5):
        key_inv, sub = jax.random.split(key_inv)
        eps_t = make_noise_samples(sub, N_inv, 1, 1, Nt, dtype=jnp.complex128)
        res   = GNLSE3D_propagate_noisy(args_inv, A_N1_3d, eps_t, sigma_inv,
                                         loss_coeff=0.0, save_as_fp32=False)
        A_out = np.array(res['field'][0, 0, :, -1])
        delta = A_out - field_clean[:, -1]
        energies.append(float(np.sum(np.abs(delta)**2) * dt))

    print(f'  {label:20s}: mean noise energy = {np.mean(energies):.3e} ± {np.std(energies):.1e}')

print('\nValues should agree within statistical fluctuations.')
print('Without sqrt(deltaZ) fix, halving deltaZ would double the noise energy.')
print('\nStep-size invariance check complete. See terminal output above.')


In [ ]:
print('=== Summary ===')
print(f'Noiseless soliton peak stability : ±{np.std(peak_clean/I_sol)*100:.3f}%')
print(f'Gradient norm (kwo space)        : {np.linalg.norm(grad_analytical):.4e}')
print(f'FD gradient check                : see rel_err printed above')
print(f'dL/d(sigma) at 3% sigma (FD)     : {dL_ds_fd:.4e}')
print()
print('FDT balance (loss_coeff = sigma²/2):')
for frac in [0.03, 0.07, 0.15]:
    sigma = frac * np.sqrt(I_sol)
    lc    = sigma**2 / 2
    print(f'  σ={frac:.0%}√I_sol: loss_coeff={lc:.3e} m⁻¹')
print()
print('Figures saved:')
for fname in sorted(os.listdir('figures')):
    if fname.endswith('.png'):
        path = os.path.join('figures', fname)
        size = os.path.getsize(path)
        print(f'  {path}  ({size//1024} KB)')
